# Notebook 06 — Explainable Priority Scoring

**Objective (from your seminar):** "Design a transparent, explainable priority scoring mechanism for undertrial case flagging."

**What this notebook does, and why:** Notebook 04 gave us a *predicted disposal time* for each case. Notebook 05 gave us a *predicted reason* each case is delayed. Neither of those, on its own, tells a court official which case to look at first. This notebook combines both — along with case severity and landmark status — into a single **priority score**, using a transparent formula rather than another black-box model, so that anyone can see exactly why one case was ranked above another.

---
### ⚠️ Before running
This notebook depends on the outputs of Notebook 04 (`cox_model.pkl`) and Notebook 05 (`dataset_with_delay_cause.csv`) — make sure both have been run successfully first. Same as those notebooks, I wrote and reasoned through this carefully but could not execute it in this sandbox (no `lifelines` / `transformers` here), so please run it on your machine and let me know if anything needs adjusting.

## Step 1 — Import libraries

In [1]:
# pandas / numpy -- for loading and working with tabular data
import pandas as pd
import numpy as np

# pickle -- to load the Cox model we already trained and saved in Notebook 04
import pickle

print("Libraries imported successfully")


Libraries imported successfully


## Step 2 — Load everything this notebook needs
Three separate outputs from earlier notebooks come together here.

In [2]:
# We need THREE pieces of information to build a priority score, and each one
# lives in a different file we've already produced:
#
#   1. structured_dataset.csv       -> case details (crime type, region, bail type, etc.)
#   2. cox_model.pkl                -> the trained Cox model, to predict how long
#                                       each case is likely to take
#   3. dataset_with_delay_cause.csv -> the predicted reason each case is delayed
#                                       (from Notebook 05)
#
# This notebook's job is to bring all three together into ONE final score.

df = pd.read_csv('../data/processed/structured_dataset.csv')

with open('../models/cox_model.pkl', 'rb') as f:
    cph = pickle.load(f)

delay_df = pd.read_csv('../data/processed/dataset_with_delay_cause.csv')[
    ['case_id', 'predicted_delay_cause', 'delay_cause_confidence']
]

print("Structured dataset:", df.shape)
print("Delay-cause dataset:", delay_df.shape)


Structured dataset: (365, 32)
Delay-cause dataset: (365, 3)


## Step 3 — Get a predicted disposal time for every case
We use the Cox model from Notebook 04 to predict a duration for ALL cases at once — this matters most for any case that is still pending, where we don't have an actual outcome yet.

In [3]:
# Before we can prioritize cases, we need each case's PREDICTED disposal time
# from the Cox model (not just its actual/estimated duration_days -- the whole
# point of a priority system is to flag cases the model expects to take long,
# which is especially useful for cases that are still pending in the future).
#
# This encoding must match EXACTLY how we encoded the data when we trained the
# model in Notebook 04, otherwise the columns won't line up.

bool_cols = ['landmark_case', 'bail_cancellation_case', 'parity_argument_used', 'has_special_law']
categorical_cols = ['crime_type', 'region_grouped', 'bail_type', 'prior_cases', 'accused_gender']

df_for_cox = df.copy()
for col in bool_cols:
    df_for_cox[col] = df_for_cox[col].astype(int)

df_encoded = pd.get_dummies(df_for_cox, columns=categorical_cols, drop_first=False)

# cph.params_.index holds the exact columns the model was trained on.
# reindex() aligns our full dataset to that same column set, filling any
# missing category-column with 0 (the same trick used in the Notebook 04
# prediction cell, just applied to ALL cases at once instead of one case).
X_for_prediction = df_encoded.reindex(columns=cph.params_.index, fill_value=0)

# predict_median() run on the WHOLE dataset at once returns one predicted
# value per row -- much faster than looping case-by-case.
df['predicted_duration'] = cph.predict_median(X_for_prediction).values

print("Predicted duration added for all cases.")
print(df[['case_id', 'crime_type', 'duration_days', 'predicted_duration']].head())


Predicted duration added for all cases.
   case_id         crime_type  duration_days  predicted_duration
0        3  Fraud or Cheating             33                42.0
1       63   Theft or Robbery             22                26.0
2       64  Fraud or Cheating             26                49.0
3       72  Fraud or Cheating             10                13.0
4       83  Fraud or Cheating             26                38.0


## Step 4 — Bring in the predicted delay cause from Notebook 05

In [4]:
# Bring in the delay-cause prediction from Notebook 05, matched by case_id
df = df.merge(delay_df, on='case_id', how='left')

# A few cases might not have a delay-cause prediction yet (e.g. if Notebook 05
# hasn't been run on the full dataset) -- we handle that gracefully instead of
# letting it silently break the scoring step later.
df['predicted_delay_cause'] = df['predicted_delay_cause'].fillna('No specific delay noted')

print(df['predicted_delay_cause'].value_counts())


predicted_delay_cause
No specific delay noted              361
Witness non-availability               3
Procedural or documentation delay      1
Name: count, dtype: int64


## Step 5 — Build 4 interpretable scoring components
Each factor is scaled to a 0–1 range so they can be combined fairly, and each one has a plain-language reason for existing — this is what makes the final score explainable rather than arbitrary.

In [5]:
# ============================================================
# WHY a WEIGHTED, EXPLAINABLE formula instead of another ML model?
# ============================================================
# Our project's own objective is "an explainable prioritization algorithm for
# highlighting the features used for determining case urgency." A black-box
# model (even a very accurate one) can't tell a court official WHY case A
# outranks case B -- a transparent formula can. So instead of training yet
# another model here, we combine a handful of interpretable factors, each
# scaled to a 0-1 range, with a clear weight showing how much it counts.

# ---- Factor 1: Predicted duration score ----
# Cases the Cox model expects to take LONGER get a higher urgency push --
# these are the cases most likely to sit in the backlog if nothing is done.
min_d, max_d = df['predicted_duration'].min(), df['predicted_duration'].max()
df['duration_score'] = (df['predicted_duration'] - min_d) / (max_d - min_d)

# ---- Factor 2: Delay-cause severity score ----
# Not all delay causes deserve equal urgency. A case stuck on "Repeated
# adjournments" or "Procedural/documentation delay" reflects a process
# bottleneck the court CAN act on -- so we weight those higher than causes
# like "Investigation pending" (largely outside the court's direct control)
# or "No specific delay noted" (no evidence of a problem to act on).
delay_cause_weights = {
    "Repeated adjournments": 1.0,
    "Procedural or documentation delay": 0.9,
    "Witness non-availability": 0.6,
    "Investigation pending": 0.4,
    "No specific delay noted": 0.1,
}
df['delay_cause_score'] = df['predicted_delay_cause'].map(delay_cause_weights).fillna(0.5)

# ---- Factor 3: Case severity score ----
# A simple, stated ranking of how serious each crime category is treated for
# prioritization purposes (this is a starting assumption -- adjust freely if
# your domain knowledge suggests a different ordering).
severity_weights = {
    "Fraud or Cheating": 1.0,
    "Theft or Robbery": 0.7,
    "Cyber Crime": 0.4,
}
df['severity_score'] = df['crime_type'].map(severity_weights).fillna(0.5)

# ---- Factor 4: Landmark case flag ----
# Landmark / precedent-setting cases often warrant closer attention.
df['landmark_score'] = df['landmark_case'].astype(int)

print("All 4 scoring components computed:")
print(df[['duration_score', 'delay_cause_score', 'severity_score', 'landmark_score']].describe())


All 4 scoring components computed:
       duration_score  delay_cause_score  severity_score  landmark_score
count      365.000000         365.000000      365.000000      365.000000
mean         0.053466           0.106301        0.704110        0.093151
std          0.099326           0.061472        0.242037        0.291042
min          0.000000           0.100000        0.400000        0.000000
25%          0.016340           0.100000        0.400000        0.000000
50%          0.020697           0.100000        0.700000        0.000000
75%          0.037037           0.100000        1.000000        0.000000
max          1.000000           0.900000        1.000000        1.000000


## Step 6 — Combine the components into one final priority score
A weighted sum, not a trained model — the weights are stated explicitly, so the formula itself IS the explanation.

In [6]:
# ============================================================
# Combine the 4 components into ONE final, explainable priority score
# ============================================================
# Weights are chosen so predicted duration and delay cause -- the two factors
# most directly tied to WHY a case is stuck -- carry the most weight, while
# case severity and landmark status act as smaller adjustments.

WEIGHTS = {
    'duration_score': 0.40,
    'delay_cause_score': 0.35,
    'severity_score': 0.15,
    'landmark_score': 0.10,
}
# NOTE: weights sum to 1.0, so the final priority_score also stays in [0, 1],
# which makes it easy to interpret (closer to 1 = more urgent).
assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9, "Weights must sum to 1.0"

df['priority_score'] = (
    WEIGHTS['duration_score']    * df['duration_score'] +
    WEIGHTS['delay_cause_score'] * df['delay_cause_score'] +
    WEIGHTS['severity_score']    * df['severity_score'] +
    WEIGHTS['landmark_score']    * df['landmark_score']
)

print("Priority score computed for all cases.")
print(df[['case_id', 'crime_type', 'predicted_duration', 'predicted_delay_cause', 'priority_score']]
      .sort_values('priority_score', ascending=False).head(10))


Priority score computed for all cases.
     case_id         crime_type  predicted_duration  \
24       199  Fraud or Cheating               925.0   
247      896   Theft or Robbery                20.0   
235      879  Fraud or Cheating               531.0   
242      890  Fraud or Cheating               428.0   
349     1177  Fraud or Cheating               428.0   
13       147  Fraud or Cheating                10.0   
243      892  Fraud or Cheating                 8.0   
81       326  Fraud or Cheating               148.0   
7         95  Fraud or Cheating               145.0   
352     1180  Fraud or Cheating               358.0   

                 predicted_delay_cause  priority_score  
24             No specific delay noted        0.685000  
247  Procedural or documentation delay        0.425664  
235            No specific delay noted        0.413322  
242            No specific delay noted        0.368442  
349            No specific delay noted        0.368442  
13           

## Step 7 — Explain a single case's score, step by step
This is the cell that makes the system genuinely explainable in practice — for your demo/viva, or for an actual court official, this shows exactly which factors pushed a case's score up or down.

In [7]:
# ============================================================
# EXPLAINABILITY: show exactly how ONE case's score was built
# ============================================================
# This is the cell you'd actually show a court official or your examiner --
# it breaks the final number down into its contributing parts, so nothing
# about the ranking is a "black box."

case_id_to_explain = df.loc[df['priority_score'].idxmax(), 'case_id']  # highest-priority case, as an example
row = df[df['case_id'] == case_id_to_explain].iloc[0]

print(f"Case ID: {case_id_to_explain}  |  Crime type: {row['crime_type']}  |  Region: {row['region_grouped']}")
print(f"Predicted duration: {row['predicted_duration']:.1f} days")
print(f"Predicted delay cause: {row['predicted_delay_cause']}")
print()
print("Score breakdown:")
for component, weight in WEIGHTS.items():
    contribution = weight * row[component]
    print(f"  {component:<20} raw={row[component]:.2f}  x  weight={weight:.2f}  =  {contribution:.3f}")
print(f"\n  TOTAL PRIORITY SCORE = {row['priority_score']:.3f}")


Case ID: 199  |  Crime type: Fraud or Cheating  |  Region: Uttar Pradesh
Predicted duration: 925.0 days
Predicted delay cause: No specific delay noted

Score breakdown:
  duration_score       raw=1.00  x  weight=0.40  =  0.400
  delay_cause_score    raw=0.10  x  weight=0.35  =  0.035
  severity_score       raw=1.00  x  weight=0.15  =  0.150
  landmark_score       raw=1.00  x  weight=0.10  =  0.100

  TOTAL PRIORITY SCORE = 0.685


## Step 8 — Save the final prioritized case list
This becomes the input for the last piece of your system architecture — the dashboard.

In [8]:
# Save the final, fully-scored dataset. This is the file a dashboard
# (the last piece of your system architecture) would read from directly.
output_cols = ['case_id', 'crime_type', 'region_grouped', 'predicted_duration',
               'predicted_delay_cause', 'duration_score', 'delay_cause_score',
               'severity_score', 'landmark_score', 'priority_score']
df[output_cols].sort_values('priority_score', ascending=False).to_csv(
    '../data/processed/prioritized_cases.csv', index=False
)
print("Saved -> data/processed/prioritized_cases.csv")


Saved -> data/processed/prioritized_cases.csv
